# 门控循环单元（GRU）
:label:`sec_gru`

在 :numref:`sec_bptt`中，
我们讨论了如何在循环神经网络中计算梯度，
以及矩阵连续乘积可以导致梯度消失或梯度爆炸的问题。
下面我们简单思考一下这种梯度异常在实践中的意义：

* 我们可能会遇到这样的情况：早期观测值对预测所有未来观测值具有非常重要的意义。
  考虑一个极端情况，其中第一个观测值包含一个校验和，
  目标是在序列的末尾辨别校验和是否正确。
  在这种情况下，第一个词元的影响至关重要。
  我们希望有某些机制能够在一个记忆元里存储重要的早期信息。
  如果没有这样的机制，我们将不得不给这个观测值指定一个非常大的梯度，
  因为它会影响所有后续的观测值。
* 我们可能会遇到这样的情况：一些词元没有相关的观测值。
  例如，在对网页内容进行情感分析时，
  可能有一些辅助HTML代码与网页传达的情绪无关。
  我们希望有一些机制来*跳过*隐状态表示中的此类词元。
* 我们可能会遇到这样的情况：序列的各个部分之间存在逻辑中断。
  例如，书的章节之间可能会有过渡存在，
  或者证券的熊市和牛市之间可能会有过渡存在。
  在这种情况下，最好有一种方法来*重置*我们的内部状态表示。

在学术界已经提出了许多方法来解决这类问题。
其中最早的方法是"长短期记忆"（long-short-term memory，LSTM）
 :cite:`Hochreiter.Schmidhuber.1997`，
我们将在 :numref:`sec_lstm`中讨论。
门控循环单元（gated recurrent unit，GRU）
 :cite:`Cho.Van-Merrienboer.Bahdanau.ea.2014`
是一个稍微简化的变体，通常能够提供同等的效果，
并且计算 :cite:`Chung.Gulcehre.Cho.ea.2014`的速度明显更快。
由于门控循环单元更简单，我们从它开始解读。

## 门控隐状态

门控循环单元与普通的循环神经网络之间的关键区别在于：
前者支持隐状态的门控。
这意味着模型有专门的机制来确定应该何时更新隐状态，
以及应该何时重置隐状态。
这些机制是可学习的，并且能够解决了上面列出的问题。
例如，如果第一个词元非常重要，
模型将学会在第一次观测之后不更新隐状态。
同样，模型也可以学会跳过不相关的临时观测。
最后，模型还将学会在需要的时候重置隐状态。
下面我们将详细讨论各类门控。

### 重置门和更新门

我们首先介绍*重置门*（reset gate）和*更新门*（update gate）。
我们把它们设计成$(0, 1)$区间中的向量，
这样我们就可以进行凸组合。
重置门允许我们控制“可能还想记住”的过去状态的数量；
更新门将允许我们控制新状态中有多少个是旧状态的副本。

我们从构造这些门控开始。 :numref:`fig_gru_1`
描述了门控循环单元中的重置门和更新门的输入，
输入是由当前时间步的输入和前一时间步的隐状态给出。
两个门的输出是由使用sigmoid激活函数的两个全连接层给出。

![在门控循环单元模型中计算重置门和更新门](../img/gru-1.svg)
:label:`fig_gru_1`

我们来看一下门控循环单元的数学表达。
对于给定的时间步$t$，假设输入是一个小批量
$\mathbf{X}_t \in \mathbb{R}^{n \times d}$
（样本个数$n$，输入个数$d$），
上一个时间步的隐状态是
$\mathbf{H}_{t-1} \in \mathbb{R}^{n \times h}$
（隐藏单元个数$h$）。
那么，重置门$\mathbf{R}_t \in \mathbb{R}^{n \times h}$和
更新门$\mathbf{Z}_t \in \mathbb{R}^{n \times h}$的计算如下所示：

$$
\begin{aligned}
\mathbf{R}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xr} + \mathbf{H}_{t-1} \mathbf{W}_{hr} + \mathbf{b}_r),\\
\mathbf{Z}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xz} + \mathbf{H}_{t-1} \mathbf{W}_{hz} + \mathbf{b}_z),
\end{aligned}
$$

其中$\mathbf{W}_{xr}, \mathbf{W}_{xz} \in \mathbb{R}^{d \times h}$
和$\mathbf{W}_{hr}, \mathbf{W}_{hz} \in \mathbb{R}^{h \times h}$是权重参数，
$\mathbf{b}_r, \mathbf{b}_z \in \mathbb{R}^{1 \times h}$是偏置参数。
请注意，在求和过程中会触发广播机制
（请参阅 :numref:`subsec_broadcasting`）。
我们使用sigmoid函数（如 :numref:`sec_mlp`中介绍的）
将输入值转换到区间$(0, 1)$。

### 候选隐状态

接下来，让我们将重置门$\mathbf{R}_t$
与 :eqref:`rnn_h_with_state`
中的常规隐状态更新机制集成，
得到在时间步$t$的*候选隐状态*（candidate hidden state）
$\tilde{\mathbf{H}}_t \in \mathbb{R}^{n \times h}$。

$$\tilde{\mathbf{H}}_t = \tanh(\mathbf{X}_t \mathbf{W}_{xh} + \left(\mathbf{R}_t \odot \mathbf{H}_{t-1}\right) \mathbf{W}_{hh} + \mathbf{b}_h),$$
:eqlabel:`gru_tilde_H`

其中$\mathbf{W}_{xh} \in \mathbb{R}^{d \times h}$
和$\mathbf{W}_{hh} \in \mathbb{R}^{h \times h}$是权重参数，
$\mathbf{b}_h \in \mathbb{R}^{1 \times h}$是偏置项，
符号$\odot$是Hadamard积（按元素乘积）运算符。
在这里，我们使用tanh非线性激活函数来确保候选隐状态中的值保持在区间$(-1, 1)$中。

与 :eqref:`rnn_h_with_state`相比，
 :eqref:`gru_tilde_H`中的$\mathbf{R}_t$和$\mathbf{H}_{t-1}$
的元素相乘可以减少以往状态的影响。
每当重置门$\mathbf{R}_t$中的项接近$1$时，
我们恢复一个如 :eqref:`rnn_h_with_state`中的普通的循环神经网络。
对于重置门$\mathbf{R}_t$中所有接近$0$的项，
候选隐状态是以$\mathbf{X}_t$作为输入的多层感知机的结果。
因此，任何预先存在的隐状态都会被*重置*为默认值。

 :numref:`fig_gru_2`说明了应用重置门之后的计算流程。

![在门控循环单元模型中计算候选隐状态](../img/gru-2.svg)
:label:`fig_gru_2`

### 隐状态

上述的计算结果只是候选隐状态，我们仍然需要结合更新门$\mathbf{Z}_t$的效果。
这一步确定新的隐状态$\mathbf{H}_t \in \mathbb{R}^{n \times h}$
在多大程度上来自旧的状态$\mathbf{H}_{t-1}$和
新的候选状态$\tilde{\mathbf{H}}_t$。
更新门$\mathbf{Z}_t$仅需要在
$\mathbf{H}_{t-1}$和$\tilde{\mathbf{H}}_t$
之间进行按元素的凸组合就可以实现这个目标。
这就得出了门控循环单元的最终更新公式：

$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1}  + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t.$$

每当更新门$\mathbf{Z}_t$接近$1$时，模型就倾向只保留旧状态。
此时，来自$\mathbf{X}_t$的信息基本上被忽略，
从而有效地跳过了依赖链条中的时间步$t$。
相反，当$\mathbf{Z}_t$接近$0$时，
新的隐状态$\mathbf{H}_t$就会接近候选隐状态$\tilde{\mathbf{H}}_t$。
这些设计可以帮助我们处理循环神经网络中的梯度消失问题，
并更好地捕获时间步距离很长的序列的依赖关系。
例如，如果整个子序列的所有时间步的更新门都接近于$1$，
则无论序列的长度如何，在序列起始时间步的旧隐状态都将很容易保留并传递到序列结束。

 :numref:`fig_gru_3`说明了更新门起作用后的计算流。

![计算门控循环单元模型中的隐状态](../img/gru-3.svg)
:label:`fig_gru_3`

总之，门控循环单元具有以下两个显著特征：

* 重置门有助于捕获序列中的短期依赖关系；
* 更新门有助于捕获序列中的长期依赖关系。

## 从零开始实现

为了更好地理解门控循环单元模型，我们从零开始实现它。
首先，我们读取 :numref:`sec_rnn_scratch`中使用的时间机器数据集：


In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

# 批次大小为32，每个样本包含35个时间步
batch_size, num_steps = 32, 35

# 使用 D2L 1.0.3 的新版 TimeMachine 数据集
data = d2l.TimeMachine(
    batch_size=batch_size,
    num_steps=num_steps
)

# 获取训练数据迭代器
train_iter = data.get_dataloader(train=True)

# 获取词表
vocab = data.vocab

# 查看数据集信息
print("词表大小：", len(vocab))

# 读取一个批次
X, Y = next(iter(train_iter))

print("输入 X 的形状：", X.shape)
print("标签 Y 的形状：", Y.shape)
print("输入 X：", X[0])
print("标签 Y：", Y[0])


### [**初始化模型参数**]

下一步是初始化模型参数。
我们从标准差为$0.01$的高斯分布中提取权重，
并将偏置项设为$0$，超参数`num_hiddens`定义隐藏单元的数量，
实例化与更新门、重置门、候选隐状态和输出层相关的所有权重和偏置。


In [ ]:
def get_params(vocab_size, num_hiddens, device):
    """
    初始化 GRU 模型的所有可学习参数。

    参数:
        vocab_size: 词表大小，也是输入和输出维度
        num_hiddens: 隐藏单元数量
        device: 计算设备（CPU 或 GPU）

    返回:
        params: 需要参与梯度更新的模型参数列表
    """

    # 输入和输出维度都等于词表大小
    num_inputs = num_outputs = vocab_size

    # 初始化权重：均值为0、标准差为0.01的正态分布
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01

    # 创建门控单元所需的三个参数
    def three():
        return (
            normal((num_inputs, num_hiddens)),   # 输入权重
            normal((num_hiddens, num_hiddens)),  # 隐状态权重
            torch.zeros(num_hiddens, device=device)  # 偏置
        )

    # 1. 更新门（Update Gate）
    # 决定保留多少上一时刻的隐藏状态
    W_xz, W_hz, b_z = three()

    # 2. 重置门（Reset Gate）
    # 决定计算候选隐状态时使用多少历史信息
    W_xr, W_hr, b_r = three()

    # 3. 候选隐状态（Candidate Hidden State）
    # 结合当前输入和重置后的历史信息
    W_xh, W_hh, b_h = three()

    # 4. 输出层
    # 将隐藏状态映射为词表中各字符的预测分数
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)

    # 5. 收集所有需要学习的参数
    params = [
        W_xz, W_hz, b_z,  # 更新门
        W_xr, W_hr, b_r,  # 重置门
        W_xh, W_hh, b_h,  # 候选隐状态
        W_hq, b_q         # 输出层
    ]

    # 启用自动求导
    for param in params:
        param.requires_grad_(True)

    return params


### 定义模型

现在我们将[**定义隐状态的初始化函数**]`init_gru_state`。
与 :numref:`sec_rnn_scratch`中定义的`init_rnn_state`函数一样，
此函数返回一个形状为（批量大小，隐藏单元个数）的张量，张量的值全部为零。


In [ ]:
def init_gru_state(batch_size, num_hiddens, device):
    """
    初始化 GRU 的隐藏状态。

    参数:
        batch_size: 批次大小，例如32
        num_hiddens: 隐藏单元数量，例如256
        device: 计算设备（CPU或GPU）

    返回:
        包含初始隐藏状态的元组
    """

    # 创建形状为 (batch_size, num_hiddens) 的全零张量
    # 每个样本对应一个独立的隐藏状态向量
    H = torch.zeros(
        (batch_size, num_hiddens),
        device=device
    )

    # 返回元组，方便与D2L模型接口保持一致
    return (H,)

现在我们准备[**定义门控循环单元模型**]，
模型的架构与基本的循环神经网络单元是相同的，
只是权重更新公式更为复杂。


In [ ]:
def gru(inputs, state, params):
    """
    GRU 的前向传播计算。

    参数:
        inputs: 输入序列，形状为 (num_steps, batch_size, vocab_size)
        state: 上一时刻的隐藏状态，包含 H 的元组
        params: GRU 模型的所有权重和偏置参数

    返回:
        outputs: 所有时间步的预测输出
        (H,): 最后一个时间步的隐藏状态
    """

    # 1. 解包模型参数
    W_xz, W_hz, b_z, \
    W_xr, W_hr, b_r, \
    W_xh, W_hh, b_h, \
    W_hq, b_q = params

    # 2. 取出初始隐藏状态
    H, = state

    # 保存每个时间步的预测结果
    outputs = []

    # 3. 遍历时间序列
    # X 的形状：(batch_size, vocab_size)
    for X in inputs:

        # 更新门 Z：决定保留多少旧隐藏状态
        Z = torch.sigmoid(
            (X @ W_xz) + (H @ W_hz) + b_z
        )

        # 重置门 R：控制计算候选隐状态时使用多少历史信息
        R = torch.sigmoid(
            (X @ W_xr) + (H @ W_hr) + b_r
        )

        # 候选隐状态：融合当前输入和经过重置门筛选的历史信息
        H_tilda = torch.tanh(
            (X @ W_xh) + ((R * H) @ W_hh) + b_h
        )

        # 更新隐藏状态：融合旧状态与候选状态
        H = Z * H + (1 - Z) * H_tilda

        # 输出层：将隐藏状态映射到词表维度
        Y = H @ W_hq + b_q

        # 保存当前时间步的预测
        outputs.append(Y)

    # 4. 拼接所有时间步的输出
    # 输出形状：(num_steps * batch_size, vocab_size)
    # H 的形状：(batch_size, num_hiddens)
    return torch.cat(outputs, dim=0), (H,)

### [**训练**]与预测

训练和预测的工作方式与 :numref:`sec_rnn_scratch`完全相同。
训练结束后，我们分别打印输出训练集的困惑度，
以及前缀“time traveler”和“traveler”的预测序列上的困惑度。


In [ ]:
# 设置模型参数
vocab_size, num_hiddens, device = len(vocab), 256, d2l.try_gpu()
num_epochs, lr = 50, 1

# 封装从零实现的GRU模型
class GRUModelScratch(nn.Module):
    def __init__(self, vocab_size, num_hiddens, device):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens

        # 注册可训练参数
        self.params = nn.ParameterList([
            nn.Parameter(p)
            for p in get_params(vocab_size, num_hiddens, device)
        ])

    def forward(self, X):
        # 将字符索引转换为独热编码
        X = nn.functional.one_hot(
            X.T.long(), self.vocab_size
        ).float()

        # 初始化隐藏状态并执行GRU前向传播
        state = init_gru_state(
            X.shape[1], self.num_hiddens, X.device
        )
        return gru(X, state, self.params)


# 创建GRU模型
model = GRUModelScratch(
    vocab_size, num_hiddens, device
).to(device)

# 定义损失函数和优化器
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

# 保存每轮训练的困惑度
perplexities = []

# 训练GRU模型
for epoch in range(num_epochs):
    model.train()
    total_loss, total_tokens = 0.0, 0

    for X, Y in train_iter:
        X, Y = X.to(device), Y.to(device)

        # 清空梯度
        optimizer.zero_grad()

        # 前向传播
        y_hat, _ = model(X)

        # 计算损失
        y = Y.T.reshape(-1)
        loss = loss_fn(y_hat, y)

        # 反向传播和梯度裁剪
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # 更新参数
        optimizer.step()

        # 累计损失
        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()

    # 计算当前轮的困惑度
    ppl = torch.exp(
        torch.tensor(total_loss / total_tokens)
    ).item()
    perplexities.append(ppl)

    # 每5轮实时更新训练曲线
    if (epoch + 1) % 5 == 0 or epoch == 0:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(
            range(1, len(perplexities) + 1),
            perplexities,
            label="Training Perplexity"
        )

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Perplexity")
        ax.set_title(f"GRU Training - Epoch {epoch + 1}/{num_epochs}")
        ax.legend()
        ax.grid(True)

        display(fig)
        plt.close(fig)

        print(f"epoch {epoch + 1}, perplexity {ppl:.2f}")

## [**简洁实现**]

高级API包含了前文介绍的所有配置细节，
所以我们可以直接实例化门控循环单元模型。
这段代码的运行速度要快得多，
因为它使用的是编译好的运算符而不是Python来处理之前阐述的许多细节。


In [ ]:
num_inputs = vocab_size              # 输入维度 = 词表大小（字符级 one-hot 维度）
gru_layer = nn.GRU(num_inputs, num_hiddens)  # 单层GRU，输入vocab_size，隐藏单元num_hiddens
model = d2l.RNNModel(gru_layer, len(vocab))  # 封装GRU + 输出全连接层（映射到词表大小）
model = model.to(device)             # 模型移到GPU/CPU
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)  # 训练：模型、数据、词表、学习率、轮数、设备

## 小结

* 门控循环神经网络可以更好地捕获时间步距离很长的序列上的依赖关系。
* 重置门有助于捕获序列中的短期依赖关系。
* 更新门有助于捕获序列中的长期依赖关系。
* 重置门打开时，门控循环单元包含基本循环神经网络；更新门打开时，门控循环单元可以跳过子序列。